In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("briscdataset/brisc2025")

print(path)

In [ ]:
import os

print(os.listdir(path))

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print(files[:10])

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

seg_path = Path(path) / "brisc2025" / "segmentation_task"

train_images = list((seg_path / "train" / "images").glob("*"))
train_masks = list((seg_path / "train" / "masks").glob("*"))

test_images = list((seg_path / "test" / "images").glob("*"))
test_masks = list((seg_path / "test" / "masks").glob("*"))

print("🧠 BRISC2025 - Segmentation Dataset")
print("=" * 50)

print(f"📁 Train Images : {len(train_images):,}")
print(f"🎭 Train Masks  : {len(train_masks):,}")
print(f"📁 Test Images  : {len(test_images):,}")
print(f"🎭 Test Masks   : {len(test_masks):,}")

print("=" * 50)
print("📂 Dataset Structure")
print("""
segmentation_task/
│
├── 🟢 train/
│   ├── 🖼️ images/
│   └── 🎭 masks/
│
└── 🔵 test/
    ├── 🖼️ images/
    └── 🎭 masks/
""")

# MRI Segmentation


In [ ]:
from pathlib import Path

base_path = Path(path) / "brisc2025" / "segmentation_task"

train_path = base_path / "train"
test_path = base_path / "test"

print("Train:", train_path)
print("Test :", test_path)

In [ ]:
import os

print("Path:", path)

for root, dirs, files in os.walk(path):
    if "segmentation_task" in root:
        print(root)
        print(files[:5])

#display imgs before modeling

In [ ]:
import os

BASE_PATH = "/kaggle/input/brisc2025/brisc2025/segmentation_task"

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")

TEST_IMG_DIR = os.path.join(BASE_PATH, "test/images")
TEST_MASK_DIR = os.path.join(BASE_PATH, "test/masks")

print("Train images:", len(os.listdir(TRAIN_IMG_DIR)))
print("Train masks :", len(os.listdir(TRAIN_MASK_DIR)))

print("Test images :", len(os.listdir(TEST_IMG_DIR)))
print("Test masks  :", len(os.listdir(TEST_MASK_DIR)))

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

img_name = sorted(os.listdir(TRAIN_IMG_DIR))[0]

mask_name = os.path.splitext(img_name)[0] + ".png"

img_path = os.path.join(TRAIN_IMG_DIR, img_name)
mask_path = os.path.join(TRAIN_MASK_DIR, mask_name)

image = Image.open(img_path)
mask = Image.open(mask_path)

print("Image:", img_name)
print("Mask :", mask_name)
print("Image size:", image.size)
print("Mask size :", mask.size)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("MRI Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Ground Truth Mask")
plt.axis("off")

plt.show()

#images list


In [ ]:
import tensorflow as tf
import os

IMG_SIZE = 224
BATCH_SIZE = 16

train_image_files = sorted([
    os.path.join(TRAIN_IMG_DIR, f)
    for f in os.listdir(TRAIN_IMG_DIR)
    if f.endswith(".jpg")
])

train_mask_files = sorted([
    os.path.join(TRAIN_MASK_DIR, f)
    for f in os.listdir(TRAIN_MASK_DIR)
    if f.endswith(".png")
])

test_image_files = sorted([
    os.path.join(TEST_IMG_DIR, f)
    for f in os.listdir(TEST_IMG_DIR)
    if f.endswith(".jpg")
])

test_mask_files = sorted([
    os.path.join(TEST_MASK_DIR, f)
    for f in os.listdir(TEST_MASK_DIR)
    if f.endswith(".png")
])

print(len(train_image_files))
print(len(train_mask_files))
print(len(test_image_files))
print(len(test_mask_files))

In [ ]:
def load_image_mask(image_path, mask_path):

    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    # Normalize image
    image = tf.cast(image, tf.float32) / 255.0

    # Load mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )

    # Convert mask to 0/1
    mask = tf.cast(mask > 0, tf.float32)

    return image, mask

#dataset proccesing

In [ ]:
train_seg_ds = tf.data.Dataset.from_tensor_slices(
    (train_image_files, train_mask_files)
)

train_seg_ds = train_seg_ds.map(
    load_image_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_seg_ds = train_seg_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
test_seg_ds = tf.data.Dataset.from_tensor_slices(
    (test_image_files, test_mask_files)
)

test_seg_ds = test_seg_ds.map(
    load_image_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_seg_ds = test_seg_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
for images, masks in train_seg_ds.take(1):

    print("Images:", images.shape)
    print("Masks :", masks.shape)

#remove 10 img to test after deployment
(unseen data)

In [ ]:
import random

random.seed(42)

indices = list(range(len(train_image_files)))
random.shuffle(indices)

deployment_indices = indices[:10]

remaining_indices = indices[10:]

deployment_image_files = [train_image_files[i] for i in deployment_indices]
deployment_mask_files  = [train_mask_files[i] for i in deployment_indices]

train_image_files = [train_image_files[i] for i in remaining_indices]
train_mask_files  = [train_mask_files[i] for i in remaining_indices]

print("Deployment images:", len(deployment_image_files))
print("Deployment masks :", len(deployment_mask_files))
print("Remaining images :", len(train_image_files))
print("Remaining masks  :", len(train_mask_files))

In [ ]:
import pandas as pd

deployment_df = pd.DataFrame({
    "image_path": deployment_image_files,
    "mask_path": deployment_mask_files
})

deployment_df.to_csv(
    "deployment_10_images.csv",
    index=False
)

deployment_df

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for i in range(5):
    image = Image.open(deployment_image_files[i])
    mask = Image.open(deployment_mask_files[i])

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.title("MRI")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(mask, cmap="gray")
    plt.title("Mask")
    plt.axis("off")

    plt.show()

# Validation Data

In [ ]:
import tensorflow as tf

full_train_ds = tf.data.Dataset.from_tensor_slices(
    (train_image_files, train_mask_files)
)

full_train_ds = full_train_ds.shuffle(
    buffer_size=len(train_image_files),
    seed=42,
    reshuffle_each_iteration=False
)

val_size = int(0.10 * len(train_image_files))

val_ds = full_train_ds.take(val_size)
train_ds_seg = full_train_ds.skip(val_size)

print("Training samples:", len(train_image_files) - val_size)
print("Validation samples:", val_size)

Batch

In [ ]:
train_ds_seg = train_ds_seg.map(
    load_image_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    load_image_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds_seg = train_ds_seg.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

In [ ]:
for images, masks in train_ds_seg.take(1):
    print("Train images:", images.shape)
    print("Train masks :", masks.shape)

for images, masks in val_ds.take(1):
    print("Val images:", images.shape)
    print("Val masks :", masks.shape)

In [ ]:
import matplotlib.pyplot as plt

for images, masks in train_ds_seg.take(1):

    plt.figure(figsize=(10, 8))

    for i in range(4):
        plt.subplot(4, 2, 2*i + 1)
        plt.imshow(images[i])
        plt.title("MRI")
        plt.axis("off")

        plt.subplot(4, 2, 2*i + 2)
        plt.imshow(masks[i].numpy().squeeze(), cmap="gray")
        plt.title("Mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu"
    )(x)

    return x


# U-NET

Input

In [ ]:

inputs = layers.Input(shape=(224, 224, 3))


Encoder

In [ ]:

c1 = conv_block(inputs, 64)
p1 = layers.MaxPooling2D((2, 2))(c1)

c2 = conv_block(p1, 128)
p2 = layers.MaxPooling2D((2, 2))(c2)

c3 = conv_block(p2, 256)
p3 = layers.MaxPooling2D((2, 2))(c3)



In [ ]:
c4 = conv_block(p3, 512)

Decoder

In [ ]:
u3 = layers.Conv2DTranspose(
    256, 2, strides=2, padding="same"
)(c4)

u3 = layers.Concatenate()([u3, c3])
c5 = conv_block(u3, 256)


u2 = layers.Conv2DTranspose(
    128, 2, strides=2, padding="same"
)(c5)

u2 = layers.Concatenate()([u2, c2])
c6 = conv_block(u2, 128)


u1 = layers.Conv2DTranspose(
    64, 2, strides=2, padding="same"
)(c6)

u1 = layers.Concatenate()([u1, c1])
c7 = conv_block(u1, 64)



output

In [ ]:
outputs = layers.Conv2D(
    1, 1, activation="sigmoid"
)(c7)


Model

In [ ]:
model = Model(inputs, outputs)

model.summary()

Binary Cross Entropy + Dice Loss

In [ ]:
import tensorflow as tf

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    dice = (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) +
        tf.reduce_sum(y_pred) +
        smooth
    )

    return 1.0 - dice


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true, y_pred
    )

    bce = tf.reduce_mean(bce)

    dice = dice_loss(y_true, y_pred)

    return bce + dice

Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth) /
        (tf.reduce_sum(y_true) +
         tf.reduce_sum(y_pred) +
         smooth)
    )

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[
        dice_coefficient,
        tf.keras.metrics.BinaryAccuracy(name="accuracy")
    ]
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds_seg,
    validation_data=val_ds,
    epochs=20
)

 In image segmentation, accuracy is not always a reliable metric because most pixels belong to the background rather than the tumor. A model can achieve very high accuracy simply by correctly predicting the background, even if its tumor segmentation is poor. Therefore, metrics such as Dice Score and IoU are more informative because they directly measure how well the predicted tumor region overlaps with the ground-truth mask.

try deployment files

In [ ]:
print(len(deployment_image_files))
print(len(deployment_mask_files))

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

image_path = deployment_image_files[4]
mask_path = deployment_mask_files[4]

# Original image
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))

image_array = np.array(image, dtype=np.float32) / 255.0

# Model prediction
input_image = np.expand_dims(image_array, axis=0)

pred = model.predict(input_image, verbose=0)

# Predicted mask
pred_mask = (pred[0, :, :, 0] > 0.5).astype(np.float32)

# Ground Truth
true_mask = Image.open(mask_path).convert("L")
true_mask = true_mask.resize((224, 224))
true_mask = np.array(true_mask)
true_mask = (true_mask > 0).astype(np.float32)


# ==========================================
# Overlay mask on original image
# ==========================================

overlay = image_array.copy()

# Make mask region red
overlay[pred_mask == 1] = [1.0, 0.0, 0.0]

# Blend original image + mask
alpha = 0.4

result = image_array.copy()

mask_area = pred_mask == 1

result[mask_area] = (
    alpha * overlay[mask_area] +
    (1 - alpha) * image_array[mask_area]
)


# ==========================================
# Display
# ==========================================

plt.figure(figsize=(8, 8))

plt.imshow(result)
plt.axis("off")
plt.title("Predicted Segmentation Overlay")

plt.show()

In [ ]:
# ==========================================
# Prepare predicted overlay
# ==========================================

overlay = image_array.copy()

# Red color for predicted mask
red = np.zeros_like(image_array)
red[:, :, 0] = 1.0

alpha = 0.4

result = image_array.copy()

mask_area = pred_mask == 1

result[mask_area] = (
    (1 - alpha) * image_array[mask_area]
    + alpha * red[mask_area]
)


# ==========================================
# Display 3 images side by side
# ==========================================

plt.figure(figsize=(15, 5))

# 1. Original Image
plt.subplot(1, 3, 1)
plt.imshow(image_array)
plt.title("Original Image")
plt.axis("off")

# 2. Ground Truth Mask
plt.subplot(1, 3, 2)
plt.imshow(true_mask, cmap="gray")
plt.title("Ground Truth Mask")
plt.axis("off")

# 3. Prediction Overlay
plt.subplot(1, 3, 3)
plt.imshow(result)
plt.title("Predicted Mask Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
intersection = np.sum(true_mask * pred_mask)

dice = (
    2 * intersection
    / (np.sum(true_mask) + np.sum(pred_mask) + 1e-7)
)

print("Dice Score:", dice)

In [ ]:
test_image_files = sorted([
    os.path.join(TEST_IMG_DIR, f)
    for f in os.listdir(TEST_IMG_DIR)
    if f.endswith(".jpg")
])

test_mask_files = sorted([
    os.path.join(TEST_MASK_DIR, f)
    for f in os.listdir(TEST_MASK_DIR)
    if f.endswith(".png")
])

print("Test images:", len(test_image_files))
print("Test masks :", len(test_mask_files))

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (test_image_files, test_mask_files)
)

test_ds = test_ds.map(
    load_image_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds = test_ds.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

In [ ]:
for images, masks in test_ds.take(1):
    print("Test images:", images.shape)
    print("Test masks :", masks.shape)

In [ ]:
iou_metric = tf.keras.metrics.BinaryIoU(
    target_class_ids=[1],
    threshold=0.5,
    name="iou"
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[
        dice_coefficient,
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        iou_metric
    ]
)

In [ ]:
test_results = model.evaluate(
    test_ds,
    return_dict=True
)

print("\nFinal Test Results:")

for name, value in test_results.items():
    print(f"{name}: {value:.4f}")

In [ ]:
model.save("model.keras")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("model.keras")